# Tokenisation: word, character, subword

A model never sees text. It sees integer token ids. Tokenisation is the rule that turns a
string into those ids. Three choices sit on a spectrum.

- Word-level: one id per word. Vocabulary is huge and unbounded, and any word not seen in
  training maps to a single unknown token.
- Character-level: one id per character. Vocabulary is tiny and nothing is unknown, but a
  sentence becomes a very long sequence.
- Subword (BPE): common pieces get their own id, rare words split into smaller pieces.
  It sits between the two extremes and handles unseen words without an unknown token.

This notebook builds byte-pair encoding by hand, then trains a library tokeniser offline,
then loads the Qwen tokeniser if the network allows.

In [ ]:
from llm_finetune import bpe

from common.dl import set_seed

set_seed(42)

example = "the lowest newer widget"
words = example.split()

word_tokens = words
char_tokens = list(example)
print(f"word-level : {len(word_tokens)} tokens -> {word_tokens}")
print(f"char-level : {len(char_tokens)} tokens -> {char_tokens}")

The same four-word string is 4 word tokens or 23 character tokens. Word tokens are short
but brittle, a new word like `widget` has no id if it was never in training. Character
tokens never fail but the sequence is nearly six times longer, and the model has to relearn
that `t` `h` `e` means `the` every time. Subword encoding is the middle path.

## From scratch: byte-pair encoding

BPE starts with every word split into characters plus an end-of-word marker `</w>`. It
counts adjacent symbol pairs across the corpus, merges the most frequent pair into one
symbol, and repeats for a fixed number of merges. The learned merges are an ordered list.
Encoding a new word replays those merges in order.

In [ ]:
corpus = (
    ["low"] * 5
    + ["lower"] * 2
    + ["newest"] * 6
    + ["widest"] * 3
)

num_merges = 10
merges = bpe.train_bpe(corpus, num_merges=num_merges)

print(f"corpus words : {len(corpus)}")
print(f"unique words : {len(set(corpus))}")
print(f"merges learnt: {len(merges)}")
for i, (left, right) in enumerate(merges, 1):
    print(f"  {i:2d}: {left!r} + {right!r} -> {left + right!r}")

The first merges fuse the pieces that repeat most. Here `t</w>`, then `st</w>`, then
`est</w>` form in order, so the common ending `est</w>` becomes one symbol. Frequent endings
earn their own token. Now encode words from the corpus and a word the corpus never contained.

In [ ]:
for word in ["newest", "lowest", "widest"]:
    pieces = bpe.encode(word, merges)
    print(f"{word:8s} -> {pieces}  ({len(pieces)} tokens)")

# A rare, unseen word still encodes, no out-of-vocabulary token needed.
unseen = "lowestest"
pieces = bpe.encode(unseen, merges)
print(f"\n{unseen:8s} -> {pieces}  ({len(pieces)} tokens)")
print("unseen word still splits into known pieces, no OOV")

`newest` collapses to 1 token because every piece was merged during training. `lowest` was
never a training word yet it still splits cleanly into 2 known pieces, and `lowestest` into
5. That is the property word-level tokenisation lacks. The vocabulary is the starting
characters plus one new symbol per merge, so it stays bounded while coverage grows.

## Library version: ByteLevel BPE, offline

The `tokenizers` library trains the same algorithm at scale. ByteLevel BPE works on raw
bytes, so it covers any Unicode input and never emits an unknown token. We train on a small
in-notebook corpus with a few hundred vocabulary slots, fully offline and seeded by the
fixed data.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

sentences = [
    "the newest widget is the lowest priced widget we stock",
    "a wider widget costs more than a lower widget",
    "newer models ship with the widest screen and the lowest weight",
    "the lowest newest widget beats the older wider one",
] * 40

tok = ByteLevelBPETokenizer()
tok.train_from_iterator(sentences, vocab_size=300, min_frequency=2)

print(f"vocab size: {tok.get_vocab_size()}")
out = tok.encode("the lowest newest widget")
print(f"ids   : {out.ids}")
print(f"pieces: {out.tokens}  ({len(out.tokens)} tokens)")

The library tokeniser keeps each common word whole, so `the lowest newest widget` is 4
pieces. The leading `Ġ` marks a space, that is how ByteLevel records word boundaries. The
from-scratch encoder, trained on only four words with a budget of 10 merges, splits the same
string into 12 pieces. A bigger corpus and a larger merge budget buy longer tokens and
shorter sequences, which the next cell makes concrete.

In [ ]:
scratch_total = sum(len(bpe.encode(w, merges)) for w in ["the", "lowest", "newest", "widget"])
library_total = len(out.tokens)
print(f"from-scratch pieces for the four words: {scratch_total}")
print(f"library pieces for the same string   : {library_total}")

## Network-gated: the Qwen tokeniser

A production tokeniser is trained on billions of words. Loading Qwen's tokeniser shows real
subword pieces from a model you will fine-tune later. This cell needs the Hugging Face hub,
so it is wrapped to skip cleanly when offline. Only network and load errors are caught, a
real bug still surfaces.

In [ ]:
try:
    from transformers import AutoTokenizer

    qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
    text = "the lowest newest widget"
    ids = qwen_tok.encode(text)
    pieces = qwen_tok.convert_ids_to_tokens(ids)
    print(f"Qwen vocab size: {qwen_tok.vocab_size}")
    print(f"ids   : {ids}")
    print(f"pieces: {pieces}  ({len(pieces)} tokens)")
except (OSError, ImportError) as err:
    print("Qwen tokenizer unavailable offline, skipping.")
    print(f"  reason: {type(err).__name__}")

## What breaks

Vocabulary size and sequence length trade off against each other. More merges give larger
tokens, so sequences get shorter, but the embedding table and output layer grow with the
vocabulary. Fewer merges keep the vocabulary small but split words into many pieces, so
sequences get longer and training costs more per sentence. BPE stops merging at a fixed
budget, the vocabulary size, and that budget is the knob you tune for the trade-off.